In [1]:
!pip install -q --upgrade ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.2/624.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 8.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.11.0 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/auto-hh/src
%load_ext autoreload
%autoreload 2

/content/drive/MyDrive/auto-hh/src


In [4]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.0 MB/s eta 0:00:00


In [5]:
import torch
from models import BiEncoder, CrossEncoder
from lib import load_vector_store, create_vector_store
from core import Retriever, Matcher
from schemas import Resume

In [6]:
MODEL_PATH = "/content/drive/MyDrive/auto-hh/models/BiEncoder"
DATA_PATH = "/content/drive/MyDrive/auto-hh/hh_dataset.csv"
INDEX_PATH = "/content/drive/MyDrive/auto-hh/faiss_index"

In [7]:
print("🔄 Загрузка би энкодера...")

bi_encoder = BiEncoder.load_trained(
    MODEL_PATH,
    "BAAI/bge-m3",
    need_attention=True,
    use_lora=True
)
bi_encoder.eval()
if torch.cuda.is_available():
    bi_encoder = bi_encoder.to('cuda')

print("✅ BiEncoder готов")

🔄 Загрузка би энкодера...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Адаптеры LoRA загружены из: /content/drive/MyDrive/auto-hh/models/BiEncoder/adapter
✅ BiEncoder готов


In [8]:
print("📦 Создание индекса...")

create_vector_store(bi_encoder, DATA_PATH, INDEX_PATH)

📦 Создание индекса...
📦 Создание FAISS индекса...
✅ Текстов: 4,499
🔢 Векторизация...
✅ Эмбеддинги: (4499, 1024)
✅ Индекс: 4,499 векторов
✅ Тексты сохранены: 4499
✅ Сохранено в: /content/drive/MyDrive/auto-hh/faiss_index


<faiss.swigfaiss_avx512.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x7b546fdcfc00> >

In [9]:
print("📥 Загрузка Cross-Encoder...")
cross_encoder = CrossEncoder("BAAI/bge-reranker-v2-m3")

📥 Загрузка Cross-Encoder...


TypeError: CrossEncoder.__init__() missing 1 required positional argument: 'device'

In [ ]:
from pathlib import Path

faiss_files = list(Path(INDEX_PATH).iterdir())
print(f"📁 Файлы в {INDEX_PATH}:")
for f in faiss_files:
    size_kb = f.stat().st_size / 1024
    print(f"   - {f.name} ({size_kb:.1f} KB)")

In [ ]:
store = load_vector_store(INDEX_PATH, bi_encoder)

print(f"✅ vacancy_texts в store: {'vacancy_texts' in store}")
print(f"✅ Количество текстов: {len(store['vacancy_texts']) if store['vacancy_texts'] else 0}")
print(f"\n📄 Пример текста [0]:")
print(store['vacancy_texts'][0][:100] if store['vacancy_texts'] else "НЕТ ТЕКСТОВ")

In [ ]:
retriever = Retriever(
    bi_encoder=bi_encoder,
    cross_encoder=cross_encoder,
    store=store,
    retrieval_top_k=10,
    final_top_k=3,
    min_score=0.0,
)

print(f"✅ Retriever готов")

In [ ]:
query = "Python разработчик Django"
results = retriever.search(query)

print(f"\n🔍 Запрос: {query}\n")
print(*results, sep='\n')
for i, res in enumerate(results, 1):
    print(f"{i}. Score: {res['score']:.4f} | ID: {res['vacancy_id']}")
    print(f"   Роль: {res['target_role']} | Grade: {res['grade']}")
    print(f"   Компания: {res['company']}\n")

In [ ]:
matcher = Matcher(retriever=retriever)

In [ ]:
resume = Resume(
    resume_id=123,
    job_title="Python разработчик",
    exp_text="3 года бэкенда",
    skills_res="Python, Django",
    about_me="Опыт разработки",
)

result = matcher.match(resume)

In [ ]:
print(result)